In [2]:
import os 
from dotenv import load_dotenv,find_dotenv
_ = load_dotenv(find_dotenv())
groq_api_key = os.environ['GROQ_API_KEY']

## Completion Model - oneshot prompt

In [12]:
from langchain_groq import ChatGroq

llmamodel_completion = ChatGroq()

## Chat completion model - conversation model


In [ ]:
from langchain_groq import ChatGroq
llmamodel_chat = ChatGroq(model='llama-3.3-70b-versatile')

## Prompts and Prompt templates

In [ ]:
#This is for completion model
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(

    'tell me a {adjective} story about {topic}'
)

llmModelPrompt = prompt_template.format(

    adjective = "curious",
    topic = "kenedy family"
)

res = llmamodel_chat.invoke(llmModelPrompt)
print(res.content)

In [ ]:
#this is for chat completion model mostly used model 
from langchain_core.prompts import ChatPromptTemplate

chat_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system","You are an {profession} expert on {topic}"),
        ("human","hello mr {profession}, can you please answer aquestion"),
        ("ai","sure!"),
        ("human ,{user_input}"),

    ]
)

messages = chat_prompt_template.format_messages(

    profession = 'historian',
    topic = 'the kennedy family',
    user_input = 'how may childrens had joshep p kennedy?'
)

res = llmamodel_chat.invoke(messages)
print(res.content)


## Few Shot Prompting

In [25]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [

    {"input": "hii!", "output":"hola!"},
    {"input":"bye!","output":"adiós!"}
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human","{input}"),
        ("ai","{output}")
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(

    example_prompt=example_prompt,
    examples=examples
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system","you are an english-spanish translator"),
        few_shot_prompt,
        ("human","{input}"),
    ]
)




## Chains

In [30]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [

    {"input": "hii!", "output":"hola!"},
    {"input":"bye!","output":"adiós!"}
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human","{input}"),
        ("ai","{output}")
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(

    example_prompt=example_prompt,
    examples=examples
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system","you are an english-spanish translator"),
        few_shot_prompt,
        ("human","{input}"),
    ]
)

#Langchain expresion language syntax 
var_chain = final_prompt | llmamodel_chat

res = var_chain.invoke(
    {
    "input": "how are you"
    }
)
print(res.content)


estoy bien, gracias. y tú?


## Output Parser

In [32]:
from langchain_groq import ChatGroq

chat_model = ChatGroq(model='llama-3.1-8b-instant')

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers.json import SimpleJsonOutputParser

json_prompt = PromptTemplate.from_template(

    "return a json object with the 'answer' key that answers the following question {question} "
)

json_parser = SimpleJsonOutputParser() #parses the output into JSON format

json_chain = json_prompt | chat_model | json_parser #chain == prompt-->model-->genereted output-->json_parser= JSON_OUTPUT

res = json_chain.invoke({"question":"what is the biggest country"})
print(res)

## Another way you can parse the output using custome classes with pydantic 

In [45]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel,Field

In [41]:
# Define the pydantic object with the desired output format

class joke(BaseModel):
    setup: str =  Field(description="question to setup a joke")
    punchline: str = Field(description="answer to resolve a joke")
    

In [ ]:
# define the parser refering to the pydantic object 

parser = JsonOutputParser(pydantic_object = joke)

#Add the parser output format instructions in the prompt definition 

prompt = PromptTemplate(

    template = "Answer the users query. \n {format_instructions}\n {query}\n",
    input_variables = ['query'],
    partial_variables = {"format_instructions": parser.get_format_instructions()},#return a json object
    
)

chain = prompt | llmamodel_chat | parser

chain.invoke({"query": "tell me a joke"})

{'setup': "Why couldn't the bicycle stand up by itself?",
 'punchline': 'Because it was two-tired.'}